# 04 — SQL Analytics: WO-1 Readiness Audit

## Purpose

WO-1 establishes the verified baseline for analytical modelling. It reconciles the existing quality registry and STAGING row counts, inspects current database structures, validates student-presentation keys, and measures assessment-join fan-out.

Canonical SQL definitions live in `sql/04_analytics/`. This notebook invokes them by query ID through `src/analytics/db.py`; SQL is not duplicated here.

## Execution controls

- Each query uses a separate PostgreSQL connection with read-only session settings, a 120-second statement timeout, and a 5-second lock timeout.
- Displayed results are capped at 500 rows. Exact counts are computed in PostgreSQL; the full `student_vle` table is not loaded into Pandas.
- RAW loading, STAGING creation, quality-registry seeding, and DDL are outside this work order.
- AN2.01 and AN2.02 depend on successful key validation in AN1.01.
- Historical counts are reference values, not evidence of the current database state.

## Work-order status

**Status:** Prepared for local execution. No live WO-1 findings have been validated yet.

## 0. Query helper and connection controls

The helper resolves canonical SQL files by stable query ID, opens a separate read-only database session for each query, and returns a bounded result table with execution time. Connection settings are read from the local environment and are never displayed.

In [ ]:
# 1. Load the project root and canonical query runner.
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "sql" / "04_analytics").exists():
    PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "sql" / "04_analytics").exists():
    raise FileNotFoundError("Open this notebook from the repository or notebooks folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.db import run_query

query_results = {}
print("Project root:", PROJECT_ROOT.name)
print("Canonical SQL folder found:", (PROJECT_ROOT / "sql" / "04_analytics").exists())

**Validation status:** The import and path checks establish whether the canonical SQL directory and query helper are available. No database result is produced by this cell.

## AN0.01 — Database identity and session safety

**Question:** Is the active connection using the intended database and read-only session settings?

**Expectation:** The database is `oulad_analytics`; read-only mode is enabled.

**Interpretation:** A database or session-setting mismatch invalidates subsequent results and must be resolved before the audit proceeds.

In [ ]:
# 1. Verify the active database and read-only session before continuing.
result = run_query("AN0.01")
query_results["AN0.01"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

row = result["data"].iloc[0]
if row["database_name"] != "oulad_analytics" or str(row["transaction_read_only"]).lower() != "on":
    raise RuntimeError("Safety gate failed. Stop the notebook and investigate the active connection.")

### Interpretation and decision — AN0.01

**Observed result:** Pending local execution.

**Decision rule:** Continue only when the database identity is correct and the session confirms read-only operation. Otherwise, suspend WO-1 and investigate the connection configuration.

## AN0.02 — Existing quality registry snapshot

**Question:** Does the live registry match the documented historical record?

**Historical reference:** 29 records and 29 distinct test codes; 25 `PASS`, 4 `REVIEW`, and 0 `FAIL`. Historical `executed_at` values may be NULL.

**Interpretation:** A difference is a reconciliation issue. This query reads the registry; it does not rerun or reseed completed engineering checks.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.02")
query_results["AN0.02"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

### Interpretation and decision — AN0.02

**Observed result:** Pending local execution.

**Decision rule:** Reconcile any differences against existing registry contents and documented execution history. Do not reconstruct missing historical timestamps or rerun seed operations as a substitute for reconciliation.

## AN0.03 — STAGING row-count reconciliation

**Question:** Do current STAGING row counts match the established baseline?

**Historical reference:** `courses` 22; `assessments` 206; `student_info` 32,593; `student_registration` 32,593; `student_assessment` 173,912; `vle` 6,364; `student_vle` 10,655,280.

**Interpretation:** PostgreSQL computes exact counts and returns seven summary rows. The large `student_vle` scan is subject to the configured timeout; no table data is transferred to Pandas.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.03")
query_results["AN0.03"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

### Interpretation and decision — AN0.03

**Observed result:** Pending local execution.

**Decision rule:** Matching counts support continuity with the historical baseline but do not establish semantic correctness. Any discrepancy requires a targeted read-only investigation before dependent analysis proceeds.

## AN0.04 — STAGING sizes and indexes

**Question:** What physical relation sizes and indexes exist in the current database?

**Expectation:** The live catalog, not assumptions about CTAS, determines which indexes exist.

**Interpretation:** Record relation sizes and existing index definitions as a performance baseline. This query creates no indexes or other database objects.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.04")
query_results["AN0.04"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

### Interpretation and decision — AN0.04

**Observed result:** Pending local execution.

**Decision rule:** Use measured sizes and existing index definitions to assess the cost of later analytical queries. Any proposal to add an index requires a separate cost-benefit review and approval.

## AN1.01 — Student key completeness and uniqueness

**Question:** Is `(code_module, code_presentation, id_student)` complete and unique in both `student_info` and `student_registration`?

**Expectation:** No key components are NULL and no duplicate key groups exist.

**Interpretation:** AN2 profile queries are valid only if both tables pass this key check.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.01")
query_results["AN1.01"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

### Interpretation and decision — AN1.01

**Observed result:** Pending local execution.

**Decision rule:** Proceed to AN1.02 and the gated AN2 queries only if both tables satisfy key completeness and uniqueness. Otherwise, investigate the affected key groups before using student-presentation grain.

## AN1.02 — Student key-set reconciliation

**Question:** Do `student_info` and `student_registration` contain the same composite-key population, and how do their distinct student counts compare?

**Expectation:** No keys are exclusive to either table if both represent the same student-presentation population.

**Interpretation:** Read this result alongside AN1.01; equal row counts alone do not prove that the key sets match.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.02")
query_results["AN1.02"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

### Interpretation and decision — AN1.02

**Observed result:** Pending local execution.

**Decision rule:** Any key-set difference requires identifying which table contains the unmatched keys and whether the difference reflects population definitions, missing records, or a data issue. Do not infer equivalence from equal totals alone.

## AN1.03 — Assessment join fan-out

**Question:** How much does joining `student_assessment` to `student_info` by `id_student` alone inflate the row count compared with the assessment-metadata path?

**Expectation:** The student-only join may multiply rows because one student can appear in multiple module presentations. The defensible route joins through `assessments` and matches the module and presentation.

**Interpretation:** A multiplier above 1 indicates join fan-out, not additional assessment records. Unmatched records must also be considered.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.03")
query_results["AN1.03"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

### Interpretation and decision — AN1.03

**Observed result:** Pending local execution.

**Decision rule:** Use the assessment-metadata path for student-presentation analysis. Record the measured fan-out and unmatched counts; do not use the student-only join for outcome metrics.

## AN2.01 — Student counts by presentation

**Question:** Do `student_info` and `student_registration` counts reconcile at module-presentation grain?

**Expectation:** Counts should reconcile when both tables represent the same validated key population.

**Execution gate:** This query runs only when AN1.01 passes for both tables.

In [ ]:
# 1. Run this profile only when both student tables passed AN1.01.
gate = query_results.get("AN1.01")
if gate is None or len(gate["data"]) != 2 or not (gate["data"]["gate_status"] == "PASS").all():
    print("SKIPPED: AN1.01 did not confirm complete, unique keys in both student tables.")
else:
    result = run_query("AN2.01")
    query_results["AN2.01"] = result
    display(result["data"])
    print(f"Query file: {result['query_file']}")
    print(f"Elapsed seconds: {result['elapsed_seconds']}")
    print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

### Interpretation and decision — AN2.01

**Observed result:** Pending local execution.

**Decision rule:** Interpret counts at module-presentation grain. Differences must be reconciled before selecting a population denominator for subsequent metrics.

## AN2.02 — Final-result distribution

**Question:** What final-result categories are observed within each module presentation, including missing outcomes?

**Expectation:** Within-presentation percentages sum to approximately 100%.

**Denominator:** All rows in `student_info` within the relevant module presentation, including missing `final_result` values.

**Interpretation:** This is a descriptive distribution; it does not establish causes of success, failure, or withdrawal.

In [ ]:
# 1. Run this profile only when both student tables passed AN1.01.
gate = query_results.get("AN1.01")
if gate is None or len(gate["data"]) != 2 or not (gate["data"]["gate_status"] == "PASS").all():
    print("SKIPPED: AN1.01 did not confirm complete, unique keys in both student tables.")
else:
    result = run_query("AN2.02")
    query_results["AN2.02"] = result
    display(result["data"])
    print(f"Query file: {result['query_file']}")
    print(f"Elapsed seconds: {result['elapsed_seconds']}")
    print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

### Interpretation and decision — AN2.02

**Observed result:** Pending local execution.

**Decision rule:** Verify denominators and percentages within each presentation. Missing outcomes remain visible and must not be silently excluded. The distribution describes observed outcomes and does not by itself explain why they occurred.

## WO-1 acceptance criteria

WO-1 remains in progress until all eligible queries execute successfully, material deviations are investigated, interpretations and decisions are documented in `docs/analysis_log.md`, local changes are reviewed with `git diff`, and the progress ledger is updated.

Only aggregate query outputs and relevant error details are retained in the analytical record. Environment files, passwords, private connection strings, and raw CSV files are excluded.